# 📈 TCS Stock Price Prediction using LSTM
### Coding Samurai Internship - Data Science Project (Level 3)
---
**Model:** Long Short-Term Memory (LSTM)  
**Stock:** TCS (Tata Consultancy Services) - NSE India  
**Data Source:** Yahoo Finance (yfinance library)

## 📦 Step 1: Install Required Libraries

In [ ]:
# Run this cell to install all required libraries
!pip install yfinance pandas numpy matplotlib scikit-learn tensorflow keras

## 📚 Step 2: Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

print('✅ All libraries imported successfully!')

## 📥 Step 3: Download TCS Stock Data

In [ ]:
# TCS ticker on NSE is 'TCS.NS'
ticker = 'TCS.NS'

# Download last 5 years of data
df = yf.download(ticker, start='2019-01-01', end='2024-12-31')

print(f'✅ Data Downloaded!')
print(f'📊 Total Records: {len(df)}')
print(f'📅 Date Range: {df.index[0].date()} to {df.index[-1].date()}')
df.head(10)

## 🔍 Step 4: Exploratory Data Analysis (EDA)

In [ ]:
# Basic info
print('=== Dataset Info ===')
print(df.info())
print('\n=== Basic Statistics ===')
print(df.describe())
print('\n=== Missing Values ===')
print(df.isnull().sum())

In [ ]:
# Plot TCS Closing Price History
plt.figure(figsize=(14, 6))
plt.plot(df['Close'], color='blue', linewidth=1.5, label='TCS Close Price')
plt.title('TCS Stock - Closing Price History (2019-2024)', fontsize=16, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Price (INR ₹)', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Plot Volume
plt.figure(figsize=(14, 4))
plt.bar(df.index, df['Volume'], color='orange', alpha=0.7, label='Volume')
plt.title('TCS Stock - Trading Volume', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Volume')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Moving Averages
df['MA_50'] = df['Close'].rolling(window=50).mean()
df['MA_200'] = df['Close'].rolling(window=200).mean()

plt.figure(figsize=(14, 6))
plt.plot(df['Close'], label='Close Price', alpha=0.8, linewidth=1)
plt.plot(df['MA_50'], label='50-Day MA', color='orange', linewidth=1.5)
plt.plot(df['MA_200'], label='200-Day MA', color='red', linewidth=1.5)
plt.title('TCS Stock - Moving Averages', fontsize=16, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Price (INR ₹)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## ⚙️ Step 5: Data Preprocessing

In [ ]:
# Use only 'Close' price for prediction
data = df[['Close']].copy()

# Normalize data between 0 and 1 (important for LSTM)
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(data)

print(f'✅ Data scaled! Shape: {scaled_data.shape}')
print(f'Min value: {scaled_data.min():.4f}')
print(f'Max value: {scaled_data.max():.4f}')

In [ ]:
# Create sequences for LSTM
# LOOK_BACK = how many past days to use for predicting next day
LOOK_BACK = 60  # Use last 60 days to predict next day

def create_sequences(data, look_back):
    X, y = [], []
    for i in range(look_back, len(data)):
        X.append(data[i-look_back:i, 0])  # Past 60 days
        y.append(data[i, 0])               # Next day price
    return np.array(X), np.array(y)

X, y = create_sequences(scaled_data, LOOK_BACK)

# Train-Test Split (80% train, 20% test)
split = int(len(X) * 0.80)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# Reshape for LSTM: (samples, timesteps, features)
X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
X_test  = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

print(f'✅ Data split done!')
print(f'Training samples : {X_train.shape[0]}')
print(f'Testing samples  : {X_test.shape[0]}')
print(f'Input shape      : {X_train.shape}')

## 🧠 Step 6: Build LSTM Model

In [ ]:
# Build the LSTM model
model = Sequential([
    # First LSTM layer
    LSTM(units=100, return_sequences=True, input_shape=(LOOK_BACK, 1)),
    Dropout(0.2),  # Prevent overfitting
    
    # Second LSTM layer
    LSTM(units=100, return_sequences=True),
    Dropout(0.2),
    
    # Third LSTM layer
    LSTM(units=50, return_sequences=False),
    Dropout(0.2),
    
    # Output layer
    Dense(units=25),
    Dense(units=1)  # Predict 1 value (next day price)
])

# Compile the model
model.compile(optimizer='adam', loss='mean_squared_error')

# Show model summary
model.summary()

## 🏋️ Step 7: Train the Model

In [ ]:
# Early stopping to avoid overfitting
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# Train the model
print('🚀 Training started...')
history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)
print('✅ Training complete!')

In [ ]:
# Plot Training Loss
plt.figure(figsize=(10, 5))
plt.plot(history.history['loss'], label='Training Loss', color='blue')
plt.plot(history.history['val_loss'], label='Validation Loss', color='orange')
plt.title('Model Training Loss', fontsize=14, fontweight='bold')
plt.xlabel('Epochs')
plt.ylabel('Loss (MSE)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 📊 Step 8: Make Predictions & Evaluate

In [ ]:
# Make predictions
train_predict = model.predict(X_train)
test_predict  = model.predict(X_test)

# Reverse scaling (get actual INR prices back)
train_predict = scaler.inverse_transform(train_predict)
test_predict  = scaler.inverse_transform(test_predict)
y_train_actual = scaler.inverse_transform(y_train.reshape(-1, 1))
y_test_actual  = scaler.inverse_transform(y_test.reshape(-1, 1))

print('✅ Predictions made!')

In [ ]:
# Evaluation Metrics
rmse = np.sqrt(mean_squared_error(y_test_actual, test_predict))
mae  = mean_absolute_error(y_test_actual, test_predict)
r2   = r2_score(y_test_actual, test_predict)

print('='*40)
print('     📊 Model Evaluation Metrics')
print('='*40)
print(f'  RMSE (Root Mean Sq. Error) : ₹{rmse:.2f}')
print(f'  MAE  (Mean Absolute Error) : ₹{mae:.2f}')
print(f'  R²   (R-Squared Score)     : {r2:.4f}')
print('='*40)

if r2 > 0.90:
    print('  🎉 Excellent model performance!')
elif r2 > 0.75:
    print('  ✅ Good model performance!')
else:
    print('  ⚠️ Model needs improvement')

In [ ]:
# Plot: Actual vs Predicted Prices
plt.figure(figsize=(16, 7))

# Plot actual prices
plt.plot(data.index[LOOK_BACK:split+LOOK_BACK], y_train_actual, 
         color='blue', linewidth=1, label='Training Data (Actual)', alpha=0.7)

plt.plot(data.index[split+LOOK_BACK:], y_test_actual, 
         color='green', linewidth=1.5, label='Test Data (Actual)')

plt.plot(data.index[split+LOOK_BACK:], test_predict, 
         color='red', linewidth=1.5, linestyle='--', label='LSTM Predicted')

plt.title('TCS Stock Price - Actual vs LSTM Predicted', fontsize=16, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Price (INR ₹)', fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Close-up: Last 100 days Actual vs Predicted
plt.figure(figsize=(14, 6))
plt.plot(y_test_actual[-100:], color='green', label='Actual Price', linewidth=2)
plt.plot(test_predict[-100:],  color='red',   label='Predicted Price', linewidth=2, linestyle='--')
plt.title('TCS Stock - Last 100 Days: Actual vs Predicted', fontsize=14, fontweight='bold')
plt.xlabel('Days')
plt.ylabel('Price (INR ₹)')
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 🔮 Step 9: Predict Future Stock Prices (Next 30 Days)

In [ ]:
# Predict next 30 days
FUTURE_DAYS = 30

# Start with last 60 days of data
last_60_days = scaled_data[-LOOK_BACK:]
future_predictions = []

current_input = last_60_days.copy()

for _ in range(FUTURE_DAYS):
    # Reshape input
    input_data = current_input.reshape(1, LOOK_BACK, 1)
    # Predict next day
    next_price = model.predict(input_data, verbose=0)
    future_predictions.append(next_price[0, 0])
    # Update input: remove oldest, add new prediction
    current_input = np.append(current_input[1:], next_price)

# Reverse scale to get actual prices
future_predictions = scaler.inverse_transform(
    np.array(future_predictions).reshape(-1, 1)
)

# Create future dates
last_date = data.index[-1]
future_dates = pd.date_range(start=last_date + pd.Timedelta(days=1), periods=FUTURE_DAYS, freq='B')

# Display future predictions
future_df = pd.DataFrame({'Date': future_dates, 'Predicted Price (₹)': future_predictions.flatten()})
future_df['Date'] = future_df['Date'].dt.date
print('📅 TCS Stock Price Predictions - Next 30 Days')
print('='*40)
print(future_df.to_string(index=False))

In [ ]:
# Plot: Historical + Future Predictions
plt.figure(figsize=(16, 7))

# Last 6 months historical
plt.plot(data.index[-180:], data['Close'][-180:], 
         color='blue', linewidth=2, label='Historical Price')

# Future predictions
plt.plot(future_dates, future_predictions, 
         color='red', linewidth=2, linestyle='--', marker='o', 
         markersize=4, label='30-Day Forecast')

# Add vertical line at today
plt.axvline(x=data.index[-1], color='gray', linestyle=':', linewidth=2, label='Today')

plt.title('TCS Stock - Historical + 30-Day LSTM Forecast', fontsize=16, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Price (INR ₹)', fontsize=12)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'\n📈 Predicted price after 30 days: ₹{future_predictions[-1][0]:.2f}')

## ✅ Step 10: Summary & Conclusion

In [ ]:
print('='*50)
print('   📋 PROJECT SUMMARY - TCS Stock Prediction')
print('='*50)
print(f'  Stock Analyzed   : TCS (NSE India)')
print(f'  Model Used       : LSTM (Deep Learning)')
print(f'  Data Period      : 2019 - 2024 (5 Years)')
print(f'  Look-back Window : {LOOK_BACK} days')
print(f'  Training Samples : {X_train.shape[0]}')
print(f'  Testing Samples  : {X_test.shape[0]}')
print('-'*50)
print(f'  RMSE  : ₹{rmse:.2f}')
print(f'  MAE   : ₹{mae:.2f}')
print(f'  R²    : {r2:.4f}')
print('-'*50)
print(f'  30-Day Forecast  : ₹{future_predictions[-1][0]:.2f}')
print('='*50)
print('\n  ⚠️ Disclaimer: This is for educational purposes only.')
print('  Stock market predictions are not financial advice.')

---
## 📌 Key Learnings

1. **Data Collection** - Used `yfinance` to fetch real TCS stock data
2. **EDA** - Analyzed price trends, volume, and moving averages  
3. **Preprocessing** - MinMaxScaler normalization + sequence creation
4. **LSTM Model** - 3 LSTM layers with Dropout to prevent overfitting
5. **Evaluation** - RMSE, MAE, R² metrics used
6. **Forecasting** - Predicted next 30 days of TCS stock prices

---
*Project by: [Your Name] | Coding Samurai Data Science Internship*